# Linear probe on CLIP features, one generator

Logistic regression on the saved CLIP ViT-L/14 features (like Ojha et al.'s linear head), trained on
`expert_train` and evaluated on `expert_val` of one generator.

In [2]:
import sys
sys.path.insert(0, "../..")  # repo root, so `data` is importable

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from data.features import load_features

FEATURE_NAME = "clip_vit-l14"
GENERATOR = "ADM"

## Data

In [3]:
X_train, rows_train = load_features(FEATURE_NAME, pool="expert_train", generator=GENERATOR)
X_val, rows_val = load_features(FEATURE_NAME, pool="expert_val", generator=GENERATOR)
y_train, y_val = rows_train.label.to_numpy(), rows_val.label.to_numpy()

print("train", X_train.shape, "fake share", y_train.mean())
print("val  ", X_val.shape, "fake share", y_val.mean())

train (10000, 768) fake share 0.5
val   (2000, 768) fake share 0.5


## Train

In [4]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,768
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


## Evaluate

In [5]:
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val)]:
    p_fake = model.predict_proba(X)[:, 1]
    predicted = (p_fake > 0.5).astype(int)
    print(f"{name:5s} accuracy {accuracy_score(y, predicted):.4f}  ROC AUC {roc_auc_score(y, p_fake):.4f}")

print("\nval confusion matrix (rows: true real/fake, columns: predicted real/fake)")
print(confusion_matrix(y_val, model.predict(X_val)))

train accuracy 0.9940  ROC AUC 0.9998
val   accuracy 0.9580  ROC AUC 0.9918

val confusion matrix (rows: true real/fake, columns: predicted real/fake)
[[952  48]
 [ 36 964]]


## Generalization to Stable Diffusion

The model above, trained only on `GENERATOR`, evaluated on `expert_val` of the Stable Diffusion generators it never saw.

In [6]:
OTHER_GENERATORS = ["stable_diffusion_v_1_4", "stable_diffusion_v_1_5"]

for generator in OTHER_GENERATORS:
    X_other, rows_other = load_features(FEATURE_NAME, pool="expert_val", generator=generator)
    y_other = rows_other.label.to_numpy()
    p_fake = model.predict_proba(X_other)[:, 1]
    predicted = (p_fake > 0.5).astype(int)
    print(f"{generator}: {len(y_other)} images, accuracy {accuracy_score(y_other, predicted):.4f}  "
          f"ROC AUC {roc_auc_score(y_other, p_fake):.4f}")
    print(confusion_matrix(y_other, predicted), "\n")

stable_diffusion_v_1_4: 2000 images, accuracy 0.7480  ROC AUC 0.9071
[[953  47]
 [457 543]] 

stable_diffusion_v_1_5: 2000 images, accuracy 0.7435  ROC AUC 0.9040
[[961  39]
 [474 526]] 



## Comparison: linear probe vs. nearest neighbours

Ojha et al. also use k-nearest neighbours with cosine distance on the same features (k = 1, 3, 5, 9):
a test image gets the majority label of its k closest training images. For kNN, ROC AUC uses the share
of fake neighbours as the score, so it's coarse for small k (for k = 1 only 0 or 1).

In [7]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

models = {"linear probe": model}
for k in [1, 3, 5, 9, 11, 13 , 15 , 17, 19, 21, 25, 50, 100, 200, 500]:
    models[f"kNN k={k}"] = KNeighborsClassifier(n_neighbors=k, metric="cosine").fit(X_train, y_train)

test_sets = {GENERATOR: (X_val, y_val)}
for generator in OTHER_GENERATORS:
    X_other, rows_other = load_features(FEATURE_NAME, pool="expert_val", generator=generator)
    test_sets[generator] = (X_other, rows_other.label.to_numpy())

results = []
for model_name, m in models.items():
    for test_name, (X, y) in test_sets.items():
        p_fake = m.predict_proba(X)[:, 1]
        predicted = (p_fake > 0.5).astype(int)
        results.append({
            "model": model_name, "test set": test_name,
            "accuracy": accuracy_score(y, predicted),
            "ROC AUC": roc_auc_score(y, p_fake),
            "real correct": (predicted[y == 0] == 0).mean(),
            "fake correct": (predicted[y == 1] == 1).mean(),
        })
results = pd.DataFrame(results)
results.round(3)

,model,test set,accuracy,ROC AUC,real correct,fake correct
0,linear probe,ADM,0.958,0.992,0.952,0.964
1,linear probe,stable_diffusion_v_1_4,0.748,0.907,0.953,0.543
2,linear probe,stable_diffusion_v_1_5,0.744,0.904,0.961,0.526
3,kNN k=1,ADM,0.752,0.752,0.680,0.823
4,kNN k=1,stable_diffusion_v_1_4,0.582,0.581,0.678,0.485
5,kNN k=1,stable_diffusion_v_1_5,0.588,0.588,0.687,0.488
6,kNN k=3,ADM,0.764,0.836,0.674,0.855
7,kNN k=3,stable_diffusion_v_1_4,0.620,0.632,0.697,0.542
8,kNN k=3,stable_diffusion_v_1_5,0.588,0.615,0.682,0.494
9,kNN k=5,ADM,0.776,0.867,0.675,0.878


In [8]:
results.pivot(index="model", columns="test set", values="accuracy").round(3)

test set,ADM,stable_diffusion_v_1_4,stable_diffusion_v_1_5
model,,,
kNN k=1,0.752,0.582,0.588
kNN k=100,0.856,0.655,0.666
kNN k=11,0.777,0.612,0.608
kNN k=13,0.792,0.614,0.612
kNN k=15,0.798,0.617,0.618
kNN k=17,0.796,0.620,0.616
kNN k=19,0.802,0.622,0.620
kNN k=200,0.861,0.664,0.686
kNN k=21,0.813,0.624,0.620
